# Phase 1 truncation-policy spike — MiniLM-L12 on v8 residuals

Tests 5 truncation policies against the Phase 1 v8 residual artifact
with frozen sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
+ a balanced logistic regression head. **Ranking only**, not
acceptance evidence.

## Inputs

One Kaggle dataset containing `residuals.jsonl` (the §0.1-conformant
Phase 1 export from `parapet-runner/runs/phase1_v8_residuals/`).
Glob-discovered under `/kaggle/input/**/residuals.jsonl`.

## Discipline

* Train: rows with `fold_id` in {0,1,2,3}.
* Val: rows with `fold_id == 4`.
* Encoder is FROZEN. Only the linear head is trained per policy.
* No legacy `schema/eval/` YAML enters this notebook. Residuals are the
  sole input.

## Policies

| policy | description |
|--------|-------------|
| head_128 | first 128 tokens |
| tail_128 | last 128 tokens |
| head_tail_64_64 | first 64 + last 64, joined with `" ... "` |
| suspicious_span_128 | densest 128-token window by attack-trigger keywords |
| full_512 | overlapping 128-token chunks (stride 64), mean-pooled embeddings |

**Runtime:** GPU recommended (T4). **Internet:** on (for HF model download).

In [ ]:
# Config knobs. Swap to bench a different candidate; the rest follows.
import glob

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RESIDUALS_GLOB = "/kaggle/input/**/residuals.jsonl"

# Train/val split: by fold_id (matches Phase 1 §0.4 grouped CV discipline).
VAL_FOLD = 4

# Recall is reported at this fixed FPR.
TARGET_FPR = 0.10

# Policies to evaluate. Drop entries here to run a subset.
POLICIES = [
    "head_128",
    "tail_128",
    "head_tail_64_64",
    "suspicious_span_128",
    "full_512",
]

# Encoder batch size. Bumped on GPU; sentence-transformers handles padding.
ENCODER_BATCH_SIZE = 64

SEED = 42

# Glob-discover the residuals JSONL. Kaggle's mount path varies
# (e.g. /kaggle/input/datasets/<user>/<slug>/...), so we search.
_matches = glob.glob(RESIDUALS_GLOB, recursive=True)
assert len(_matches) == 1, f"expected exactly one residuals match, got: {_matches}"
RESIDUALS_PATH = _matches[0]

# Hard guard: refuse legacy eval paths under any circumstance.
_FORBIDDEN = ("schema/eval/l1_holdout", "schema/eval/challenges")
_norm = RESIDUALS_PATH.replace('\\', '/')
assert not any(s in _norm for s in _FORBIDDEN), \
    f"REFUSING: residuals path looks like a legacy eval YAML: {RESIDUALS_PATH!r}"

print(f"MODEL_ID        = {MODEL_ID}")
print(f"RESIDUALS_PATH  = {RESIDUALS_PATH}")
print(f"VAL_FOLD        = {VAL_FOLD}")
print(f"TARGET_FPR      = {TARGET_FPR}")
print(f"POLICIES        = {POLICIES}")

In [ ]:
# Hardware fingerprint mirrors scripts/hardware_fingerprint.py inline so
# the notebook does not depend on parapet/ being on the Kaggle filesystem.
import hashlib, json, os, platform, shutil, subprocess

def _sha256_text(t):
    return hashlib.sha256(t.encode('utf-8', errors='replace')).hexdigest() if t else None

def _read(p):
    try: return open(p, encoding='utf-8', errors='replace').read()
    except OSError: return None

def _run(cmd):
    if shutil.which(cmd[0]) is None: return None
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10, check=False)
    except (OSError, subprocess.SubprocessError): return None
    return r.stdout if r.returncode == 0 else None

_cpuinfo = _read('/proc/cpuinfo') if platform.system()=='Linux' else None
_lscpu   = _run(['lscpu'])         if platform.system()=='Linux' else None
_nvsmi   = _run(['nvidia-smi', '-q', '-x'])
try:
    import torch as _torch
    _torch_ver = _torch.__version__
    _cuda_avail = bool(_torch.cuda.is_available())
    _cuda_dev = _torch.cuda.get_device_name(0) if _cuda_avail else None
except ImportError:
    _torch_ver, _cuda_avail, _cuda_dev = None, False, None

HARDWARE_FINGERPRINT = {
    'platform': platform.system(),
    'platform_release': platform.release(),
    'platform_machine': platform.machine(),
    'python_version': platform.python_version(),
    'cpu_model': platform.processor() or None,
    'cpu_count_logical': os.cpu_count(),
    'cpuinfo_sha256': _sha256_text(_cpuinfo),
    'lscpu_sha256': _sha256_text(_lscpu),
    'nvsmi_sha256': _sha256_text(_nvsmi),
    'cuda_device': _cuda_dev,
    'cuda_available': _cuda_avail,
    'torch_version': _torch_ver,
}

_fp_blob = json.dumps(HARDWARE_FINGERPRINT, sort_keys=True).encode('utf-8')
HARDWARE_FINGERPRINT_SHA256 = hashlib.sha256(_fp_blob).hexdigest()

with open('/kaggle/working/hardware_fingerprint.json', 'w') as _f:
    json.dump(HARDWARE_FINGERPRINT, _f, indent=2, sort_keys=True)

print(f'hardware_fingerprint_sha256 = {HARDWARE_FINGERPRINT_SHA256}')
print(json.dumps({k: v for k, v in HARDWARE_FINGERPRINT.items() if not k.endswith('_sha256')},
                 indent=2, sort_keys=True))

In [ ]:
# Kaggle T4 image typically has sentence-transformers, transformers, and
# scikit-learn pre-installed. Pin minimums to fail fast if that ever changes.
!pip install -q "sentence-transformers>=3.0" "transformers>=4.40" "scikit-learn>=1.3"

## Truncation policies (inlined from `parapet_runner/truncation.py`)

Source-of-truth lives at `parapet-runner/parapet_runner/truncation.py` in
the parapet repo. Inlined here unmodified at notebook-build time. Round-trip
diff against the .py file should be empty.

In [ ]:
"""Truncation policies for the Phase 1 truncation-policy spike.

Each policy takes a tokenized input (a list of token ids for the full text,
EXCLUDING special tokens) and returns a (possibly shortened) list of token
ids. The caller wraps the result with [CLS]/[SEP] (or whatever the
encoder requires).

Policies (per direction.md "What's still open"):

    head_128             keep first 128 tokens
    tail_128             keep last 128 tokens
    head_tail_64_64      keep first 64 + last 64 tokens (concatenated)
    suspicious_span_128  best 128-token window by attack-trigger keyword density
    full_512             keep first 512 tokens (oracle baseline)

The suspicious-span heuristic is intentionally simple: it scores token
positions by counts of attack-trigger keywords in their surrounding
window, picks the densest window, and falls back to head_128 if no
triggers fire. This is a ranking-only proxy, not a final routing
heuristic.
"""

from __future__ import annotations

from typing import Sequence


# Curated attack-trigger lexicon. EN-heavy; the heuristic is a proxy used
# only for policy ranking. Multilingual coverage is intentionally light;
# if suspicious-span underperforms on non-EN, that itself is a useful
# signal about whether naive keyword spotting can drive routing.
ATTACK_TRIGGERS: tuple[str, ...] = (
    # instruction_override
    "ignore", "disregard", "forget", "previous instructions",
    "above instructions", "instead",
    # roleplay_jailbreak
    "dan", "developer mode", "roleplay", "pretend you are",
    "act as", "from now on you", "jailbreak",
    # meta_probe
    "system prompt", "your instructions", "your guidelines",
    "what were you told", "show me your",
    # exfiltration
    "print all", "list all", "reveal", "output your", "dump",
    # indirect_injection
    "the document says", "embedded instruction", "according to the text",
    # obfuscation
    "base64", "rot13", "decode", "encoded",
    # constraint_bypass
    "no rules", "no restrictions", "without limitation",
    "anything is allowed", "override",
)


# ---------------------------------------------------------------------------
# Token-level policies
# ---------------------------------------------------------------------------


def head_n(token_ids: Sequence[int], n: int) -> list[int]:
    """Keep the first n tokens."""
    if n <= 0:
        return []
    return list(token_ids[:n])


def tail_n(token_ids: Sequence[int], n: int) -> list[int]:
    """Keep the last n tokens."""
    if n <= 0:
        return []
    return list(token_ids[-n:])


def head_tail(token_ids: Sequence[int], head: int, tail: int) -> list[int]:
    """Keep the first ``head`` + last ``tail`` tokens.

    If the input is shorter than head+tail, just return the input unchanged
    (no synthesizing extra tokens).
    """
    if head <= 0 and tail <= 0:
        return []
    if len(token_ids) <= head + tail:
        return list(token_ids)
    return list(token_ids[:head]) + list(token_ids[-tail:])


# ---------------------------------------------------------------------------
# Suspicious-span heuristic
# ---------------------------------------------------------------------------


def find_trigger_positions(text: str, triggers: Sequence[str] = ATTACK_TRIGGERS) -> list[int]:
    """Return character offsets where any trigger keyword starts (case-insensitive).

    Pure-Python lowercase substring scan. Doesn't tokenize; just locates
    where in the source text triggers appear. The caller maps these into
    token positions later.
    """
    if not text:
        return []
    text_lower = text.lower()
    positions: list[int] = []
    for trigger in triggers:
        t = trigger.lower()
        start = 0
        while True:
            idx = text_lower.find(t, start)
            if idx == -1:
                break
            positions.append(idx)
            start = idx + 1
    positions.sort()
    return positions


def suspicious_span_window(
    token_ids: Sequence[int],
    char_offsets: Sequence[tuple[int, int]],
    text: str,
    *,
    window: int,
    triggers: Sequence[str] = ATTACK_TRIGGERS,
) -> list[int]:
    """Pick the ``window``-token slice with the highest trigger density.

    Args:
        token_ids:    list of token ids (special tokens already stripped).
        char_offsets: per-token (start_char, end_char) into ``text``.
                      Same length as token_ids. From the fast tokenizer's
                      offset_mapping with special tokens dropped.
        text:         the original text used for trigger scanning.
        window:       window size in tokens.
        triggers:     attack-trigger lexicon.

    Falls back to head_n(token_ids, window) when no triggers match.
    """
    if window <= 0 or not token_ids:
        return []
    if len(token_ids) <= window:
        return list(token_ids)
    if len(char_offsets) != len(token_ids):
        # Defensive: if offsets aren't aligned, fall back rather than skew.
        return head_n(token_ids, window)

    trigger_chars = find_trigger_positions(text, triggers)
    if not trigger_chars:
        return head_n(token_ids, window)

    # For every token, count how many triggers fall within its window.
    # Two-pointer scan: token positions are sorted by start char ascending,
    # trigger positions are sorted ascending too.
    n = len(token_ids)
    best_start = 0
    best_score = -1
    # Window covers tokens [start, start+window), which spans chars
    # [char_offsets[start][0], char_offsets[start+window-1][1]).
    for start in range(0, n - window + 1):
        win_start_char = char_offsets[start][0]
        win_end_char = char_offsets[start + window - 1][1]
        score = sum(1 for c in trigger_chars if win_start_char <= c < win_end_char)
        if score > best_score:
            best_score = score
            best_start = start
    if best_score <= 0:
        # Triggers existed but none aligned to any window — fall back.
        return head_n(token_ids, window)
    return list(token_ids[best_start:best_start + window])


# ---------------------------------------------------------------------------
# Policy registry
# ---------------------------------------------------------------------------


POLICIES: tuple[str, ...] = (
    "head_128",
    "tail_128",
    "head_tail_64_64",
    "suspicious_span_128",
    "full_512",
)


def apply_policy(
    policy: str,
    *,
    token_ids: Sequence[int],
    char_offsets: Sequence[tuple[int, int]] | None = None,
    text: str | None = None,
    triggers: Sequence[str] = ATTACK_TRIGGERS,
) -> list[int]:
    """Apply a named policy to token_ids. char_offsets/text required for suspicious_span_128."""
    if policy == "head_128":
        return head_n(token_ids, 128)
    if policy == "tail_128":
        return tail_n(token_ids, 128)
    if policy == "head_tail_64_64":
        return head_tail(token_ids, 64, 64)
    if policy == "suspicious_span_128":
        if char_offsets is None or text is None:
            raise ValueError(
                "suspicious_span_128 requires char_offsets and text"
            )
        return suspicious_span_window(
            token_ids, char_offsets, text, window=128, triggers=triggers,
        )
    if policy == "full_512":
        return head_n(token_ids, 512)
    raise ValueError(f"unknown policy {policy!r}; expected one of {POLICIES}")

In [ ]:
# Helpers mirror parapet-runner/scripts/phase1_truncation_policy_spike.py.
# Kept inline so the notebook is self-contained.

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np

CHUNK_SIZE = 128
CHUNK_STRIDE = 64
FULL_512_TOKEN_BUDGET = 512


def load_residuals(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def split_by_fold(rows, val_fold):
    train, val = [], []
    for r in rows:
        if r.get('fold_id') == val_fold:
            val.append(r)
        else:
            train.append(r)
    return train, val


def label_to_int(lab):
    return 1 if lab == 'malicious' else 0


def build_truncated_strings(text, token_ids, offsets, *, policy):
    if policy == 'full_512':
        budget = min(len(token_ids), FULL_512_TOKEN_BUDGET)
        if budget <= CHUNK_SIZE:
            return [text]
        chunks = []
        start = 0
        while start < budget:
            end = min(start + CHUNK_SIZE, budget)
            chunk_text = text[offsets[start][0]:offsets[end - 1][1]]
            if chunk_text.strip():
                chunks.append(chunk_text)
            if end >= budget:
                break
            start += CHUNK_STRIDE
        return chunks or [text]
    truncated_ids = apply_policy(
        policy, token_ids=token_ids, char_offsets=offsets, text=text,
    )
    if not truncated_ids:
        return [text[:1] or ' ']
    if policy == 'head_128':
        end_char = offsets[len(truncated_ids) - 1][1]
        return [text[:end_char]]
    if policy == 'tail_128':
        start_char = offsets[len(token_ids) - len(truncated_ids)][0]
        return [text[start_char:]]
    if policy == 'head_tail_64_64':
        head_n = min(64, len(token_ids))
        tail_n = min(64, len(token_ids) - head_n)
        if tail_n <= 0:
            return [text[:offsets[head_n - 1][1]]]
        head_text = text[:offsets[head_n - 1][1]]
        tail_text = text[offsets[len(token_ids) - tail_n][0]:]
        return [head_text + ' ... ' + tail_text]
    if policy == 'suspicious_span_128':
        n_trunc = len(truncated_ids)
        for s in range(0, len(token_ids) - n_trunc + 1):
            if token_ids[s:s + n_trunc] == truncated_ids:
                return [text[offsets[s][0]:offsets[s + n_trunc - 1][1]]]
        return [text[:offsets[min(n_trunc, len(offsets)) - 1][1]]]
    raise ValueError(f"unknown policy {policy!r}")

In [ ]:
rows = load_residuals(RESIDUALS_PATH)
print(f"Loaded {len(rows):,} residual rows")

train_rows, val_rows = split_by_fold(rows, VAL_FOLD)
print(f"Train (fold != {VAL_FOLD}): {len(train_rows):,} rows")
print(f"Val   (fold == {VAL_FOLD}): {len(val_rows):,} rows")

from collections import Counter
_train_lab = Counter(r.get('label') for r in train_rows)
_val_lab   = Counter(r.get('label') for r in val_rows)
print(f"Train labels: {dict(_train_lab)}")
print(f"Val labels:   {dict(_val_lab)}")

_val_lang = Counter(r.get('language') for r in val_rows)
_val_lang_mal = Counter()
for r in val_rows:
    if r.get('label') == 'malicious':
        _val_lang_mal[r.get('language')] += 1
print(f"Val per-lang n / n_malicious:")
for _l, _n in _val_lang.most_common():
    print(f'  {_l}: n={_n} mal={_val_lang_mal.get(_l, 0)}')

In [ ]:
import torch
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = SentenceTransformer(MODEL_ID)
_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(_DEVICE)
model.eval()
print(f'Encoder loaded on {_DEVICE}')

def tokenize_fn(text):
    out = tokenizer(
        text or ' ',
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=False,
    )
    offsets = [(int(s), int(e)) for s, e in out['offset_mapping']]
    return list(map(int, out['input_ids'])), offsets

def encoder(texts):
    if not texts:
        return np.zeros((0, 384), dtype=np.float32)
    emb = model.encode(
        texts,
        batch_size=ENCODER_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False,
        device=_DEVICE,
    )
    return emb.astype(np.float32)

In [ ]:
# Per-policy encoding. For full_512 we encode multiple chunks per row and
# mean-pool. Embeddings cached in memory (no disk cache on Kaggle).
import time

def encode_rows_for_policy(rows, policy, *, label):
    embeddings = []
    n = len(rows)
    t0 = time.time()
    for i, r in enumerate(rows):
        text = r.get('content') or ''
        token_ids, offsets = tokenize_fn(text)
        if not token_ids:
            embeddings.append(encoder([' '])[0])
            continue
        chunks = build_truncated_strings(text, token_ids, offsets, policy=policy)
        chunk_emb = encoder(chunks)
        if chunk_emb.shape[0] > 1:
            embeddings.append(chunk_emb.mean(axis=0))
        else:
            embeddings.append(chunk_emb[0])
        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 1e-3)
            print(f'  [{policy}/{label}] {i + 1}/{n} rows ({rate:.0f}/sec)')
    print(f'  [{policy}/{label}] done in {time.time() - t0:.1f}s')
    return np.vstack(embeddings).astype(np.float32)

ALL_EMBEDDINGS = {}  # policy -> {'train': arr, 'val': arr}
for _policy in POLICIES:
    print(f'\n=== Encoding {_policy} ===')
    ALL_EMBEDDINGS[_policy] = {
        'train': encode_rows_for_policy(train_rows, _policy, label='train'),
        'val':   encode_rows_for_policy(val_rows,   _policy, label='val'),
    }

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

train_labels = np.array([label_to_int(r.get('label') or 'benign') for r in train_rows])
val_labels   = np.array([label_to_int(r.get('label') or 'benign') for r in val_rows])

def recall_at_fpr(y_true, scores, target_fpr):
    fpr, tpr, thr = roc_curve(y_true, scores)
    eligible = fpr <= target_fpr
    if not eligible.any():
        return 0.0, 0.0, float('inf')
    idx = int(np.argmax(np.where(eligible, tpr, -1)))
    return float(tpr[idx]), float(fpr[idx]), float(thr[idx])

RESULTS = {}
for _policy in POLICIES:
    train_emb = ALL_EMBEDDINGS[_policy]['train']
    val_emb   = ALL_EMBEDDINGS[_policy]['val']
    clf = LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=SEED,
        solver='liblinear',
    )
    clf.fit(train_emb, train_labels)
    scores = clf.predict_proba(val_emb)[:, 1]
    if len(np.unique(val_labels)) < 2:
        recall, fpr, thr, auc = 0.0, 0.0, float('inf'), float('nan')
    else:
        recall, fpr, thr = recall_at_fpr(val_labels, scores, TARGET_FPR)
        auc = float(roc_auc_score(val_labels, scores))

    # Per-language slice
    from collections import defaultdict
    by_lang = defaultdict(list)
    for i, r in enumerate(val_rows):
        by_lang[str(r.get('language') or 'unknown')].append(i)
    per_lang = {}
    for _lang, _idxs in by_lang.items():
        idxs = np.array(_idxs)
        y_l = val_labels[idxs]
        s_l = scores[idxs]
        if len(np.unique(y_l)) < 2:
            per_lang[_lang] = {
                'n': int(len(y_l)), 'n_malicious': int(y_l.sum()),
                'recall_at_fpr': float('nan'), 'roc_auc': float('nan'),
            }
        else:
            r_l, f_l, _ = recall_at_fpr(y_l, s_l, TARGET_FPR)
            per_lang[_lang] = {
                'n': int(len(y_l)), 'n_malicious': int(y_l.sum()),
                'recall_at_fpr': r_l, 'achieved_fpr': f_l,
                'roc_auc': float(roc_auc_score(y_l, s_l)),
            }

    RESULTS[_policy] = {
        'recall_at_fpr': recall,
        'achieved_fpr': fpr,
        'threshold': thr,
        'roc_auc': auc,
        'n_val': int(len(val_labels)),
        'n_val_malicious': int(val_labels.sum()),
        'per_language': per_lang,
    }
    print(f'{_policy:<22} recall@FPR={recall:.3f} (achieved {fpr:.3f})  AUC={auc:.3f}')

In [ ]:
# Rank by recall@FPR (primary) then AUC (tiebreak).
ranked = sorted(
    RESULTS.items(),
    key=lambda kv: (-kv[1]['recall_at_fpr'], -kv[1]['roc_auc']),
)

print('\n=== Ranking ===')
print(f'{"rank":>4}  {"policy":<22}  {"recall@FPR":>12}  {"AUC":>8}')
for i, (policy, r) in enumerate(ranked):
    print(f'{i+1:>4}  {policy:<22}  {r["recall_at_fpr"]:>12.3f}  {r["roc_auc"]:>8.3f}')

import datetime
summary = {
    'schema_version': 1,
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'model_id': MODEL_ID,
    'target_fpr': TARGET_FPR,
    'val_fold': VAL_FOLD,
    'n_train': len(train_rows),
    'n_val':   len(val_rows),
    'hardware_fingerprint_sha256': HARDWARE_FINGERPRINT_SHA256,
    'residuals_path': RESIDUALS_PATH,
    'spike_scope': 'ranking only — NOT acceptance evidence',
    'ranking': [
        {
            'rank': i + 1,
            'policy': policy,
            'recall_at_fpr': r['recall_at_fpr'],
            'achieved_fpr': r['achieved_fpr'],
            'roc_auc': r['roc_auc'],
            'n_val_malicious': r['n_val_malicious'],
        }
        for i, (policy, r) in enumerate(ranked)
    ],
    'per_policy': RESULTS,
}

with open('/kaggle/working/truncation_spike_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('\nSummary written to /kaggle/working/truncation_spike_summary.json')
print('Decision rules (from direction.md):')
print('  head_128 within ~1-2 recall points of full_512 -> use head_128')
print('  head_tail_64_64 recovers suffix attacks -> prefer over head-only')
print('  full_512 materially beats all 128-token policies -> MiniLM is slow-lane only')
print('  suspicious_span_128 beats both -> mechanical scan selects what L2 sees')

In [ ]:
# Per-language detail per policy. Languages with <2 classes show NaN.
print(f'{"policy":<22}  {"lang":<4}  {"n":>4}  {"n_mal":>5}  {"r@FPR":>8}  {"AUC":>6}')
for policy, r in RESULTS.items():
    for lang, m in sorted(r['per_language'].items()):
        print(f'{policy:<22}  {lang:<4}  {m["n"]:>4}  {m["n_malicious"]:>5}  '
              f'{m.get("recall_at_fpr", float("nan")):>8.3f}  '
              f'{m.get("roc_auc", float("nan")):>6.3f}')